In [11]:
import json, os
from pathlib import Path
import pandas as pd

def load_model_results_per_task(cache_dir, default_prompt=None):
    model_results = []

    cache_dir_path = Path(cache_dir)
    for model_cache_dir_path in cache_dir_path.iterdir():
        if model_cache_dir_path.is_dir():
    
        
            project_dirs = [x for x in model_cache_dir_path.iterdir() if x.is_dir()]
            for experiments_project_dir in project_dirs:
                 if experiments_project_dir.exists():
                    for experiment_dir_path in (experiments_project_dir / "experiments").iterdir():
                        if experiment_dir_path.is_dir():
                            # print(experiment_dir_path)
                            task_file_path = [x for x in experiment_dir_path.iterdir() if x.suffix == '.json' and x.stem != "model_meta"][0]
                            model_meta_path = experiment_dir_path / 'model_meta.json'
            
                            model_meta = json.load(open(model_meta_path, 'r'))
                            scores = json.load(open(task_file_path, 'r'))['scores']
        
                            for split, split_val in scores.items():
                                # print(model_meta['experiment_kwargs'])
                                parsed_result_dict = {
                                        "main_score": split_val[0]['main_score'],
                                        "task": task_file_path.stem,
                                        "model": model_cache_dir_path.stem
                                    }
                                
        
                                if task_file_path.stem in model_meta['experiment_kwargs']['prompts']:
                                    parsed_result_dict['prompt'] = model_meta['experiment_kwargs']['prompts'][task_file_path.stem]
                                else:
                                    parsed_result_dict['prompt'] = model_meta['experiment_kwargs']['prompts'][task_file_path.stem + "-query"]
        
                                parsed_result_dict['is_baseline'] = parsed_result_dict['prompt'] == default_prompt
                                model_results.append(parsed_result_dict)
                        
                            
    return model_results
                    

In [6]:
prompts_results = load_model_results_per_task(
    "../../data/cache_data_customassignment_human_as_well/mteb_cache/results/",
)

In [10]:
# rename models for consistency

for i in range(len(prompts_results)):
    prompts_results[i]['model'] = prompts_results[i]['model'].replace("__", "/")
    if prompts_results[i]['model'] == "Qwen/Qwen3-Embedding-0":
        prompts_results[i]['model'] = "Qwen/Qwen3-Embedding-0.6B"
    if prompts_results[i]['model'] == 'BAAI/bge-base-en-v1':
        prompts_results[i]['model'] = "BAAI/bge-base-en-v1.5"
    if prompts_results[i]['model'] == "BAAI/bge-large-en-v1":
        prompts_results[i]['model'] = "BAAI/bge-large-en-v1.5"
    if prompts_results[i]['model'] == "BAAI/bge-small-en-v1":
        prompts_results[i]['model'] = "BAAI/bge-small-en-v1.5"
    if prompts_results[i]['model'] == "KaLM-Embedding/KaLM-embedding-multilingual-mini-instruct-v2":
        prompts_results[i]['model'] = "KaLM-Embedding/KaLM-embedding-multilingual-mini-instruct-v2.5"




In [8]:
human_prompts = json.load(open("../../src/manual_prompts/y_prompts.json", 'r'))

In [9]:
human_prompts[0]

{'task': 'MIRACLRetrievalHardNegatives.v2',
 'task_description': 'retrieval',
 'generated_prompt': 'Encode the following query for retrieval:'}

In [14]:
human_prompts = [x['generated_prompt'] for x in human_prompts]

In [12]:
df = pd.DataFrame(prompts_results)

In [15]:
df['is_human'] = df['prompt'].apply(lambda x: x in human_prompts)

In [16]:
df = df[df['model'].isin([
    "BAAI/bge-base-en-v1.5", "BAAI/bge-small-en-v1.5", "BAAI/bge-large-en-v1.5"
])]

In [17]:
df['is_human'].value_counts()

is_human
False    576
True     360
Name: count, dtype: int64